# Train Models with training set A and different number of Monte Carlo samples in underlying model

In [ ]:
import pandas as pd
from DerivModel import FeedForwardNetwork, train_model
from DerivPlots import scatter_plot
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
import torch.nn.functional as F
from tqdm.notebook import tqdm, trange
import numpy as np
import matplotlib.pyplot as plt

## Load and Prepare the dataset A with multiple samples

In [ ]:
filename_root = 'test_basket_mt'

In [ ]:
df1 = pd.concat(
    [pd.read_csv(f"{filename_root}{i}.csv") for i in range(20)],
    ignore_index=False
)

# Columns that need cleaning
cols_to_fix = ["maturity", "option_value", "error_estimate", "process_time", "samples"]

for col in cols_to_fix:
    if col in df1.columns:
        df1[col] = (
            df1[col]
            .astype(str)              # ensure string
            .str.strip("[]")         # remove brackets
            .astype(float)           # convert to numeric
        )

In [ ]:
df1e6 = df1[df1['samples'] == 100000]
df1e5 = df1[df1['samples'] == 10000]
df1e4 = df1[df1['samples'] == 1000]

In [ ]:
df1e6 = df1e6.drop('samples', axis=1)
df1e5 = df1e5.drop('samples', axis=1)
df1e4 = df1e4.drop('samples', axis=1)

## Train model w 1e6 samples

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
y = df1e6['option_value']

In [ ]:
X = df1e6.drop(['option_value', 'process_time', 'error_estimate'], axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.01) 

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train.to_numpy().copy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 4096
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [ ]:
test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test.to_numpy().copy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
epochs = 200
lr = 0.001

In [ ]:
modelA = FeedForwardNetwork(input_size=28, 
                            num_hidden_layers=4, 
                            neurons_per_layer=300).to(device)
optimizer = optim.Adam(modelA.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
modelA_out = train_model(modelA, data_loader, test_data_loader, loss_fn, optimizer, epochs)

## Train model w 1e5 samples

In [ ]:
yB = df1e5['option_value']

XB = df1e5.drop(['option_value', 'process_time', 'error_estimate'], axis=1)

X_trainB, X_testB, y_trainB, y_testB = train_test_split(XB, yB, test_size=0.01) 
scalerB = StandardScaler()
X_trainB = scalerB.fit_transform(X_trainB)
X_testB = scalerB.transform(X_testB)

torch_tensorB = torch.from_numpy(X_trainB).float().to(device)
y_tensorB = torch.from_numpy(y_trainB.to_numpy().copy()).float().to(device)
datasetB = TensorDataset(torch_tensorB, y_tensorB)
batch_size = 4096
data_loaderB = DataLoader(datasetB, batch_size=batch_size, shuffle=True, drop_last=True)

test_torch_tensorB = torch.from_numpy(X_testB).float().to(device)
test_y_tensorB = torch.from_numpy(y_testB.to_numpy().copy()).float().to(device)
test_datasetB = TensorDataset(test_torch_tensorB, test_y_tensorB)
test_data_loaderB = DataLoader(test_datasetB, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
modelB = FeedForwardNetwork(input_size=28, 
                            num_hidden_layers=4, 
                            neurons_per_layer=300).to(device)
optimizerB = optim.Adam(modelB.parameters(), lr=lr)
loss_fnB = torch.nn.MSELoss()

In [ ]:
model_out_B = train_model(modelB, data_loaderB, test_data_loaderB, loss_fnB, optimizerB, epochs)

## Train model w 1e4 samples¶

In [ ]:
yC = df1e4['option_value']

XC = df1e4.drop(['option_value', 'process_time', 'error_estimate'], axis=1)

X_trainC, X_testC, y_trainC, y_testC = train_test_split(XC, yC, test_size=0.01) 
scalerC = StandardScaler()
X_trainC = scalerC.fit_transform(X_trainC)
X_testC = scalerC.transform(X_testC)

torch_tensorC = torch.from_numpy(X_trainC).float().to(device)
y_tensorC = torch.from_numpy(y_trainC.to_numpy().copy()).float().to(device)
datasetC = TensorDataset(torch_tensorC, y_tensorC)
batch_size = 4096
data_loaderC = DataLoader(datasetC, batch_size=batch_size, shuffle=True, drop_last=True)

test_torch_tensorC = torch.from_numpy(X_testC).float().to(device)
test_y_tensorC = torch.from_numpy(y_testC.to_numpy().copy()).float().to(device)
test_datasetC = TensorDataset(test_torch_tensorC, test_y_tensorC)
test_data_loaderC = DataLoader(test_datasetC, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
modelC = FeedForwardNetwork(input_size=28, 
                            num_hidden_layers=4, 
                            neurons_per_layer=300).to(device)
optimizerC = optim.Adam(modelC.parameters(), lr=lr)
loss_fnC = torch.nn.MSELoss()

In [ ]:
model_out_C = train_model(modelC, data_loaderC, test_data_loaderC, loss_fnC, optimizerC, epochs)

## Load and Prepare the accuracy test dataset

In [ ]:
filename_root_acc = 'test_basket_accuracy'

df1acc = pd.concat(
    [pd.read_csv(f"{filename_root}{i}.csv") for i in range(20)],
    ignore_index=False
)

# Columns that need cleaning
cols_to_fix = ["maturity", "option_value", "error_estimate", "process_time", "samples"]

for col in cols_to_fix:
    if col in df1acc.columns:
        df1acc[col] = (
            df1acc[col]
            .astype(str)              # ensure string
            .str.strip("[]")         # remove brackets
            .astype(float)           # convert to numeric
        )

df1acc1e7 = df1acc[df1acc['samples'] == 100000]

df1acc1e7 = df1acc1e7.drop('samples', axis=1)

In [ ]:
yacc = df1acc1e7['option_value']

Xacc = df1acc1e7.drop(['option_value', 'process_time', 'error_estimate'], axis=1)

X_acc = scaler.transform(Xacc)

test_torch_tensoracc = torch.from_numpy(X_acc).float().to(device)
test_y_tensoracc = torch.from_numpy(yacc.to_numpy().copy()).float().to(device)
test_datasetacc = TensorDataset(test_torch_tensoracc, test_y_tensoracc)
test_data_loaderacc = DataLoader(test_datasetacc, batch_size=1000, shuffle=False, drop_last=True)

In [ ]:
X_accB = scalerB.transform(Xacc)

test_torch_tensoraccB = torch.from_numpy(X_accB).float().to(device)
test_y_tensoraccB = torch.from_numpy(yacc.to_numpy().copy()).float().to(device)
test_datasetaccB = TensorDataset(test_torch_tensoraccB, test_y_tensoraccB)
test_data_loaderaccB = DataLoader(test_datasetaccB, batch_size=1000, shuffle=False, drop_last=True)

In [ ]:
X_accC = scalerC.transform(Xacc)

test_torch_tensoraccC = torch.from_numpy(X_accC).float().to(device)
test_y_tensoraccC = torch.from_numpy(yacc.to_numpy().copy()).float().to(device)
test_datasetaccC = TensorDataset(test_torch_tensoraccC, test_y_tensoraccC)
test_data_loaderaccC = DataLoader(test_datasetaccC, batch_size=1000, shuffle=False, drop_last=True)

## Evaluate the models

In [ ]:
modelA.eval()
test_mse_listA = list()
A_errors = np.empty(0)
with torch.no_grad():
    for batch_X, batch_y in test_data_loaderacc:
        test_outputs = modelA(batch_X)
        mse = F.mse_loss(test_outputs.squeeze(), batch_y).item()
        A_errors = np.append(A_errors, test_outputs.squeeze().cpu().numpy() - batch_y.cpu().numpy() )
        test_mse_listA.append(mse)

In [ ]:
test_avg_mse_A = sum(test_mse_listA) / len(test_mse_listA)
print(test_avg_mse_A)

In [ ]:
scatter_plot(test_outputs, batch_y, 'modelA_scatter')

In [ ]:
x_max = 1.5
x_min = -1.5
bin_edges = np.linspace(x_min, x_max, num=40)

In [ ]:
# Create a histogram plot
plt.hist(A_errors, bins=bin_edges, color='black', alpha=0.7)
plt.xlabel('errors')
plt.xlim([x_min, x_max])
plt.savefig('model1e6errors.png', dpi=300)
plt.show()

In [ ]:
modelB.eval()
test_mse_listB = list()
B_errors = np.empty(0)
with torch.no_grad():
    for batch_X, batch_y in test_data_loaderaccB:
        test_outputs = modelB(batch_X)
        mse = F.mse_loss(test_outputs.squeeze(), batch_y).item()
        B_errors = np.append(B_errors, test_outputs.squeeze().cpu().numpy() - batch_y.cpu().numpy() )
        test_mse_listB.append(mse)
                
test_avg_mse_B = sum(test_mse_listB) / len(test_mse_listB)
print(test_avg_mse_B)

In [ ]:
scatter_plot(test_outputs, batch_y, 'modelB_scatter')

In [ ]:
# Create a histogram plot
plt.hist(B_errors, bins=bin_edges, color='black', alpha=0.7)
plt.xlabel('errors')
plt.xlim([x_min, x_max])
plt.savefig('model1e5errors.png', dpi=300)
plt.show()

In [ ]:
modelC.eval()
test_mse_listC = list()
C_errors = np.empty(0)
with torch.no_grad():
    for batch_X, batch_y in test_data_loaderaccC:
        test_outputs = modelC(batch_X)
        mse = F.mse_loss(test_outputs.squeeze(), batch_y).item()
        C_errors = np.append(C_errors, test_outputs.squeeze().cpu().numpy() - batch_y.cpu().numpy() )
        test_mse_listC.append(mse)
                
test_avg_mse_C = sum(test_mse_listC) / len(test_mse_listC)
print(test_avg_mse_C)

In [ ]:
scatter_plot(test_outputs, batch_y, 'modelB_scatter')

In [ ]:
# Create a histogram plot
plt.hist(C_errors, bins=bin_edges, color='black', alpha=0.7)
plt.xlabel('errors')
plt.xlim([x_min, x_max])
plt.savefig('model1e4errors.png', dpi=300)
plt.show()

In [ ]:
df1acc100000 = df1acc[df1acc['samples'] == 100000]
df1acc10000 = df1acc[df1acc['samples'] == 10000]
df1acc1000 = df1acc[df1acc['samples'] == 1000]

In [ ]:
df1acc100000op = df1acc100000['option_value'].to_numpy()
df1acc10000op = df1acc10000['option_value'].to_numpy()
df1acc1000op = df1acc1000['option_value'].to_numpy()
df1acc10000000op = df1acc1e7['option_value'].to_numpy()

In [ ]:
df1acc100000err = df1acc100000op - df1acc10000000op
df1acc10000err = df1acc10000op - df1acc10000000op
df1acc1000err = df1acc1000op - df1acc10000000op

In [ ]:
# Create a histogram plot
plt.hist(df1acc100000err, bins=bin_edges, color='grey', alpha=0.7)
plt.xlabel('errors')
plt.xlim([x_min, x_max])
plt.savefig('MC100000errors.png', dpi=300)
plt.show()

In [ ]:
# Create a histogram plot
plt.hist(df1acc10000err, bins=bin_edges, color='gray', alpha=0.7)
plt.xlabel('errors')
plt.xlim([x_min, x_max])
plt.savefig('MC10000errors.png', dpi=300)
plt.show()

In [ ]:
# Create a histogram plot
plt.hist(df1acc1000err, bins=bin_edges, color='gray', alpha=0.7)
plt.xlabel('errors')
plt.xlim([x_min, x_max])
plt.savefig('MC1000errors.png', dpi=300)
plt.show()

## Train a model for longer

In [ ]:
modelD = FeedForwardNetwork(input_size=28, 
                            num_hidden_layers=4, 
                            neurons_per_layer=300).to(device)
optimizer = optim.Adam(modelD.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
modelD_out = train_model(modelD, data_loader, test_data_loader, loss_fn, optimizer, 2000)

In [ ]:
modelD.eval()
test_mse_listD = list()
D_errors = np.empty(0)
with torch.no_grad():
    for batch_X, batch_y in test_data_loaderacc:
        test_outputs = modelD(batch_X)
        mse = F.mse_loss(test_outputs.squeeze(), batch_y).item()
        D_errors = np.append(D_errors, test_outputs.squeeze().cpu().numpy() - batch_y.cpu().numpy() )
        test_mse_listD.append(mse)
                
test_avg_mse_D = sum(test_mse_listD) / len(test_mse_listD)
print(test_avg_mse_D)

In [ ]:
scatter_plot(test_outputs, batch_y, 'modelD_scatter')

In [ ]:
# Create a histogram plot
plt.hist(D_errors, bins=bin_edges, color='black', alpha=0.7)
plt.xlabel('errors')
plt.xlim([x_min, x_max])
plt.savefig('modelDrrors.png', dpi=300)
plt.show()

In [ ]:
torch.save(modelA, 'modelAMC.pt')

In [ ]:
torch.save(modelB, 'modelBMC.pt')
torch.save(modelC, 'modelCMC.pt')
torch.save(modelD, 'modelDMC.pt')

## Train a model using a larger dataset

In [ ]:
filename_root = 'test_basket_big'

df1 = pd.concat(
    [pd.read_csv(f"{filename_root}{i}.csv") for i in range(20)],
    ignore_index=False
)

# Columns that need cleaning
cols_to_fix = ["maturity", "option_value", "error_estimate", "process_time", "samples"]

for col in cols_to_fix:
    if col in df1.columns:
        df1[col] = (
            df1[col]
            .astype(str)              # ensure string
            .str.strip("[]")         # remove brackets
            .astype(float)           # convert to numeric
        )

dfbig = df1[df1['samples'] == 100000]

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

y = dfbig['option_value']

X = dfbig.drop(['Unnamed: 0', 'option_value', 'process_time', 'error_estimate', 'samples'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.01) 

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train.to_numpy().copy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 4096
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test.to_numpy().copy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
X

In [ ]:
epochs = 2
lr = 0.001

modelbig = FeedForwardNetwork(input_size=28, 
                            num_hidden_layers=4, 
                            neurons_per_layer=300).to(device)
optimizer = optim.Adam(modelbig.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

modelbig_out = train_model(modelbig, data_loader, test_data_loader, loss_fn, optimizer, epochs)

In [ ]:
modelbig.eval()
test_mse_listbig = list()
big_errors = np.empty(0)
with torch.no_grad():
    for batch_X, batch_y in test_data_loaderacc:
        test_outputs = modelbig(batch_X)
        mse = F.mse_loss(test_outputs.squeeze(), batch_y).item()
        big_errors = np.append(big_errors, test_outputs.squeeze().cpu().numpy() - batch_y.cpu().numpy() )
        test_mse_listbig.append(mse)
                
test_avg_mse_big = sum(test_mse_listbig) / len(test_mse_listbig)
print(test_avg_mse_big)

In [ ]:
scatter_plot(test_outputs, batch_y, 'modelbig_scatter')

In [ ]:
# Create a histogram plot
plt.hist(big_errors, bins=bin_edges, color='black', alpha=0.7)
plt.xlabel('errors')
plt.xlim([x_min, x_max])
plt.savefig('modelbigrrors.png', dpi=300)
plt.show()

In [ ]:
torch.save(modelbig, 'modelbigMC.pt')